### Recovered symbols vs. number of waves

Fix an amplitude interval $[a_{\min}, a_{\max}]$ and give the least squares method $n$ waves with amplitudes spread evenly over it. We work directly on the recovered arrays $\hat\ell(k)$ and $\hat n(k)$ (no polynomial fit, so the same analysis works for non-local symbols such as Whitham).

Two questions:
1. How does the error in $\hat\ell,\hat n$ depend on the number of waves?
2. **Up to which $k$ are the symbols accurate?** Instead of fixing a $k_{\max}$ we use the full positive spectrum, and define the accuracy cut-off $k^*(\text{tol})$ as the largest $k$ such that the error is below `tol` for *every* wavenumber up to $k^*$.

Errors are absolute: $|\hat\ell-\ell|$ and $|\hat n - n|$ (relative error in $\ell$ is misleading near zero crossings of $\ell$).

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().parent.parent
sys.path.insert(0,str(root))

from camera.camera_file import Camera
from camera.spectralCamera import *
from camera.pdeCamera import *
from camera.lsmCamera import LSM
from solvers.help import *

from waves.waveGenCamera import KdVPacket, WhithamPacket

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#### Setup

In [ ]:
L   = 2*2*np.pi
N   = 2**7
dt  = 1e-3
T   = 0.5
c1  = Camera(L, N, dt)

# --- Which equation? Packet generates the waves, ell_theory / n_theory_fn are the true symbols
Packet      = KdVPacket              # WhithamPacket
ell_theory  = kdVDispersion          # whithamDispersion
n_theory_fn = n_theory               # same nonlinearity for both: n = 1/2

# --- Amplitude interval and numbers of waves
A_MIN, A_MAX = 0.5, 2.0
N_WAVES = [1, 2, 3, 4, 6, 8, 12, 16]

# --- Error tolerances used to define the accuracy cut-off k*
TOLS = [1e-1, 1e-2, 1e-3]

#### Helpers

All arrays are on the positive wavenumbers $k>0$ (sorted ascending, symbols assumed symmetric; $k=0$ is NaN in the LSM).

In [ ]:
def wave_amps(n, a_min=A_MIN, a_max=A_MAX):
    return np.array([(a_min + a_max)/2]) if n == 1 else np.linspace(a_min, a_max, n)

def run_lsm(n, a_min=A_MIN, a_max=A_MAX):
    amps  = wave_amps(n, a_min, a_max)
    means = np.zeros_like(amps)
    ps    = L*np.ones_like(amps)
    H     = Packet(P_list=ps, amp_list=amps, mean_list=means, camera=c1, T=T).mainloop()
    lsm   = LSM(eta_list=H, camera=c1)
    lsm.lsm()
    return lsm

pos = np.where(c1.k > 0)[0]
pos = pos[np.argsort(c1.k[pos])]     # indices of k>0 in ascending k
kp  = c1.k[pos]

ell_th = ell_theory(kp)
n_th   = n_theory_fn(kp)

def extract(lsm):
    return dict(
        ell      = lsm.ell[pos].real,
        n        = lsm.n[pos].real,
        cond     = lsm.cond[pos],
        col_norm = lsm.col_norm[pos, 0],      # size of the |H_k| data column
    )

def errors(r):
    return np.abs(r['ell'] - ell_th), np.abs(r['n'] - n_th)

def cutoff(err, tol):
    # Largest k* such that err(k) <= tol for all 0 < k <= k*. NaN counts as a failure.
    # Returns 0 if the very first mode already fails.
    bad = ~(err <= tol)
    if not bad.any():
        return kp[-1]                        # accurate on the entire spectrum
    i = np.argmax(bad)                       # first failing index
    return kp[i-1] if i > 0 else 0.0

#### Run the experiment

In [ ]:
results = {}
for nw in N_WAVES:
    results[nw] = extract(run_lsm(nw))
    print(f"n_waves = {nw:3d} done")

#### Recovered symbols and errors over the whole spectrum

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
for nw, r in results.items():
    e_ell, e_n = errors(r)
    ax[0,0].plot(kp, r['ell'], lw=1, label=f"{nw}")
    ax[0,1].plot(kp, r['n'],   lw=1, label=f"{nw}")
    ax[1,0].semilogy(kp, e_ell, lw=1)
    ax[1,1].semilogy(kp, e_n,   lw=1)

ax[0,0].plot(kp, ell_th, 'k:', lw=2, label="theory")
ax[0,1].plot(kp, n_th,   'k:', lw=2, label="theory")
ax[0,0].set_title(r"$\hat\ell$");           ax[0,1].set_title(r"$\hat n$")
ax[1,0].set_title(r"$|\hat\ell-\ell|$");    ax[1,1].set_title(r"$|\hat n-n|$")
for tol in TOLS:
    ax[1,0].axhline(tol, color='gray', ls='--', lw=0.5)
    ax[1,1].axhline(tol, color='gray', ls='--', lw=0.5)
for a in ax.ravel(): a.grid(True, which='both')
for a in ax[1]: a.set_xlabel("k")
ax[0,1].set_ylim(0, 1)               # ell/n blow up in the noisy tail; keep the plot readable
ax[0,0].legend(title="waves", ncol=2, fontsize=8)
plt.tight_layout(); plt.show()

#### Accuracy cut-off $k^*$

For each number of waves: the largest $k^*$ such that $|\hat\ell-\ell|\le$ tol (resp. $|\hat n-n|\le$ tol) for all $k\le k^*$. The Nyquist wavenumber is $k=\pi/\Delta x$ = `kp[-1]`, which is what the value saturates at if the whole spectrum is accurate.

In [ ]:
rows = []
for nw, r in results.items():
    e_ell, e_n = errors(r)
    row = {"n_waves": nw}
    for tol in TOLS:
        row[f"k*_ell (tol={tol:g})"] = cutoff(e_ell, tol)
        row[f"k*_n   (tol={tol:g})"] = cutoff(e_n,   tol)
    rows.append(row)
kstar = pd.DataFrame(rows).set_index("n_waves")
print(f"Nyquist k = {kp[-1]:.2f}")
kstar.style.format("{:.2f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for tol in TOLS:
    ax[0].semilogx(kstar.index, kstar[f"k*_ell (tol={tol:g})"], 'o-', label=f"tol = {tol:g}")
    ax[1].semilogx(kstar.index, kstar[f"k*_n   (tol={tol:g})"], 'o-', label=f"tol = {tol:g}")
ax[0].set_title(r"cut-off $k^*$ for $\ell$"); ax[1].set_title(r"cut-off $k^*$ for $n$")
for a in ax:
    a.axhline(kp[-1], color='k', ls=':', lw=0.8)
    a.set_xlabel("number of waves"); a.grid(True, which='both')
ax[0].set_ylabel("$k^*$"); ax[0].legend()
plt.tight_layout(); plt.show()

#### Where does the accuracy come from?

The LSM solves a $2\times2$-type problem independently at each $k$. Two diagnostics per mode: the size of the data column $|H_k|$ (how much signal the waves carry at $k$) and the condition number of the two-column matrix (how well $\ell$ and $n$ can be separated at $k$). The recovered symbol should break down where the signal hits noise level or the conditioning blows up.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for nw, r in results.items():
    ax[0].semilogy(kp, r['col_norm'], lw=1, label=f"{nw}")
    ax[1].semilogy(kp, r['cond'],     lw=1)
ax[0].set_title(r"data size $\|H_k\|$"); ax[1].set_title("condition number")
for a in ax:
    a.set_xlabel("k"); a.grid(True, which='both')
ax[0].legend(title="waves", ncol=2, fontsize=8)
plt.tight_layout(); plt.show()

#### Does the width of the interval matter?

Fixed number of waves, different amplitude intervals, to separate "more waves" from "more diverse amplitudes".

In [ ]:
NW_FIXED  = 8
intervals = [(1.0, 1.0001), (0.9, 1.1), (0.75, 1.25), (0.5, 1.5), (0.25, 2.0)]

rows, width_res = [], {}
for lo, hi in intervals:
    r = extract(run_lsm(NW_FIXED, lo, hi))
    width_res[(lo, hi)] = r
    e_ell, e_n = errors(r)
    row = {"interval": f"[{lo}, {hi}]"}
    for tol in TOLS:
        row[f"k*_ell (tol={tol:g})"] = cutoff(e_ell, tol)
        row[f"k*_n   (tol={tol:g})"] = cutoff(e_n,   tol)
    rows.append(row)
kstar_w = pd.DataFrame(rows).set_index("interval")
kstar_w.style.format("{:.2f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for (lo, hi), r in width_res.items():
    e_ell, e_n = errors(r)
    ax[0].semilogy(kp, e_ell, lw=1, label=f"[{lo}, {hi}]")
    ax[1].semilogy(kp, e_n,   lw=1)
ax[0].set_title(rf"$|\hat\ell-\ell|$, {NW_FIXED} waves"); ax[1].set_title(rf"$|\hat n-n|$, {NW_FIXED} waves")
for a in ax:
    a.set_xlabel("k"); a.grid(True, which='both')
ax[0].legend(fontsize=8)
plt.tight_layout(); plt.show()